# 01 - Prune all calibration datasets in parallel

Launches **three** EC2 spot GPU boxes simultaneously — one per calibration
benchmark — each running the full WANDA calibration pipeline at sparsity
levels [0, 20, 40, 60, 80]%:

* **GSM8K** (`math:gsm8k:main`) — uses the native 7 473-example train split.
* **HumanEval+** (`coding:evalplus/humanevalplus:test`) — seeded 80/20 split
  over the 164-problem test set (no native train split).
* **ARC-Challenge** (`mcq:allenai/ai2_arc:ARC-Challenge`) — uses the native
  1 119-example train split.

Artifacts land at `s3://<bucket>/pruning_artifacts/<run_id>/` and the three
URIs are written to `experiment_config.json` in this directory so that
notebook 2 (`02_eval_all.ipynb`) can pick them up automatically.

Notebook `01_setup_aws.ipynb` in `aws_tutorial/` must have run first to
provision the S3 bucket and EC2 instance profile.


In [1]:
import os
import sys
from pathlib import Path

# Locate repo root regardless of where the notebook is opened from.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env", override=False)
except ImportError:
    pass

print("REPO_ROOT =", REPO_ROOT)
print("AWS_PROFILE =", os.environ.get("AWS_PROFILE"))
print("AWS_REGION  =", os.environ.get("AWS_REGION"))


REPO_ROOT = /var/home/three-kingdoms/work/pruning-metrics
AWS_PROFILE = rengz
AWS_REGION  = us-east-1


## Configuration

* `BASE_MODEL_ID`: any HF causal LM. Defaults to Qwen2-72B.
* `CALIBRATION_SPECS`: list of `(label, adapter_spec)` pairs defining the
  three calibration benchmarks. Labels are used as keys in
  `experiment_config.json`.
* `PRUNING_LEVELS`: sparsity percentages to record in the calibration
  manifest (the actual pruning is re-applied deterministically by notebooks 2
  and 3 of the tutorial, or by `02_eval_all.ipynb`).
* All three calibration jobs share the same `SPLIT_SEED`, `TRAIN_FRAC`, and
  `INSTANCE_TYPE_PRIORITY` / `REGION_PRIORITY` lists.


In [2]:
import json

AWS_PROFILE = os.environ.get("AWS_PROFILE", "rengz")
RESULTS_BUCKET = os.environ.get(
    "RESULTS_BUCKET", "pruning-metrics-results-414266451290"
)

BASE_MODEL_ID = "Qwen/Qwen2-72B"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Each entry is (label, calibration_dataset_spec).
CALIBRATION_SPECS = [
    ("gsm8k",         "math:gsm8k:main"),
    ("humaneval",     "coding:evalplus/humanevalplus:test"),
    ("arc_challenge", "mcq:allenai/ai2_arc:ARC-Challenge"),
]

PRUNING_LEVELS = [0, 20, 40, 60, 80]
SPLIT_SEED = 65320
TRAIN_FRAC = 0.8
MAX_CALIBRATION_SAMPLES = 0   # 0 = full train split
MAX_CALIBRATION_TOKENS = 512

INSTANCE_TYPE_PRIORITY = ["p4de.24xlarge", "p5.48xlarge", "p4d.24xlarge"]
REGION_PRIORITY = ["us-east-1", "us-west-2", "us-east-2"]

print(json.dumps({
    "BASE_MODEL_ID": BASE_MODEL_ID,
    "CALIBRATION_SPECS": CALIBRATION_SPECS,
    "PRUNING_LEVELS": PRUNING_LEVELS,
    "RESULTS_BUCKET": RESULTS_BUCKET,
}, indent=2))


{
  "BASE_MODEL_ID": "Qwen/Qwen2-72B",
  "CALIBRATION_SPECS": [
    [
      "gsm8k",
      "math:gsm8k:main"
    ],
    [
      "humaneval",
      "coding:evalplus/humanevalplus:test"
    ],
    [
      "arc_challenge",
      "mcq:allenai/ai2_arc:ARC-Challenge"
    ]
  ],
  "PRUNING_LEVELS": [
    0,
    20,
    40,
    60,
    80
  ],
  "RESULTS_BUCKET": "pruning-metrics-results-414266451290"
}


## Find spot capacity

Queries `describe_spot_price_history` across the priority lists and returns
sorted `(region, AZ, instance_type)` candidates. All three calibration jobs
use the same first candidate; re-run this cell and retry the launch cell if
you hit `InsufficientInstanceCapacity`.


In [3]:
from pruning_metrics.notebook_helpers import find_capacity

candidates = find_capacity(
    regions=tuple(REGION_PRIORITY),
    instance_types=tuple(INSTANCE_TYPE_PRIORITY),
    aws_profile=AWS_PROFILE,
)
assert candidates, "No spot capacity found in the priority list."
chosen = candidates[0]
print(f"Top candidate: {chosen['region']} {chosen['availability_zone']} "
      f"{chosen['instance_type']} @ ${chosen['spot_price_usd_per_hour']:.4f}/h")
print(f"\nAll candidates ({len(candidates)} total):")
for c in candidates[:5]:
    print(f"  {c['region']} {c['availability_zone']} {c['instance_type']}"
          f" ${c['spot_price_usd_per_hour']:.4f}/h")


Top candidate: us-east-1 us-east-1d p4de.24xlarge @ $12.8718/h

All candidates (29 total):
  us-east-1 us-east-1d p4de.24xlarge $12.8718/h
  us-east-1 us-east-1c p4de.24xlarge $21.2857/h
  us-east-1 us-east-1f p5.48xlarge $11.6817/h
  us-east-1 us-east-1b p5.48xlarge $11.7115/h
  us-east-1 us-east-1e p5.48xlarge $15.9910/h


## Launch all three calibration runners

Submits all three `RunInstances` calls back-to-back (each takes ~2 s). The
boxes start in parallel. Each uploads its artifact to
`s3://<bucket>/pruning_artifacts/<run_id>/` before shutting down.


In [4]:
from pruning_metrics.notebook_helpers import launch_runner, render_run_id_default

calibration_launches = []  # list of (label, LaunchedRun)

for label, cal_spec in CALIBRATION_SPECS:
    run_id = render_run_id_default()
    runner_env = {
        "BASE_MODEL_ID": BASE_MODEL_ID,
        "CALIBRATION_DATASET_SPEC": cal_spec,
        "PRUNING_LEVELS": ",".join(str(lv) for lv in PRUNING_LEVELS),
        "SPLIT_SEED": SPLIT_SEED,
        "TRAIN_FRAC": TRAIN_FRAC,
        "MAX_CALIBRATION_SAMPLES": MAX_CALIBRATION_SAMPLES,
        "MAX_CALIBRATION_TOKENS": MAX_CALIBRATION_TOKENS,
    }
    launched = launch_runner(
        runner="pruning_calibration",
        runner_env=runner_env,
        region=chosen["region"],
        availability_zone=chosen["availability_zone"],
        instance_type=chosen["instance_type"],
        max_spot_price=float(chosen["max_bid_usd_per_hour"]),
        results_bucket=RESULTS_BUCKET,
        results_prefix="pruning_artifacts",
        run_id=run_id,
        aws_profile=AWS_PROFILE,
        hf_token=HF_TOKEN,
        name_tag=f"pruning-metrics-calibration-{label}",
    )
    calibration_launches.append((label, launched))
    print(f"[{label}] launched {launched.instance_id} run_id={launched.run_id}")
    print(f"  artifact will land at: {launched.results_uri}")


[gsm8k] launched i-09998a70347ecff76 run_id=20260506T144103Z-b08812
  artifact will land at: s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144103Z-b08812/
[humaneval] launched i-0307539dde110abd6 run_id=20260506T144135Z-991a93
  artifact will land at: s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144135Z-991a93/
[arc_challenge] launched i-092456fc6d4165f3d run_id=20260506T144208Z-122c3c
  artifact will land at: s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144208Z-122c3c/


## Wait for all three calibrations to finish

Uses `ThreadPoolExecutor` so all three S3 polls run concurrently. Each
thread blocks until `wanda_stats.pt` (the heaviest artifact) appears under
the run prefix. Calibration on `p4de.24xlarge` typically takes 30–60 min
per run (mostly HF model download + WANDA forward pass over the train split).


In [6]:
import concurrent.futures
import time

from pruning_metrics.notebook_helpers import wait_for_artifact, list_results

def _wait_calibration(label, launched):
    key = f"pruning_artifacts/{launched.run_id}/wanda_stats.pt"
    print(f"[{label}] waiting for {key} ...")
    head = wait_for_artifact(
        bucket=RESULTS_BUCKET,
        key=key,
        aws_profile=AWS_PROFILE,
        poll_seconds=30.0,
        timeout_seconds=60 * 60 * 3,
    )
    print(f"[{label}] DONE — wanda_stats.pt size={head['ContentLength']:,} bytes")
    return head

t0 = time.monotonic()
with concurrent.futures.ThreadPoolExecutor(max_workers=len(calibration_launches)) as pool:
    futs = {
        pool.submit(_wait_calibration, label, launched): (label, launched)
        for label, launched in calibration_launches
    }
    for fut in concurrent.futures.as_completed(futs):
        label, launched = futs[fut]
        try:
            fut.result()
        except Exception as exc:
            print(f"[{label}] ERROR: {exc}")

elapsed = time.monotonic() - t0
print(f"\nAll calibrations complete in {elapsed / 60:.1f} min.")

# List all artifacts for each run.
for label, launched in calibration_launches:
    prefix = f"pruning_artifacts/{launched.run_id}"
    print(f"\n[{label}] {launched.results_uri}")
    for entry in list_results(RESULTS_BUCKET, prefix, aws_profile=AWS_PROFILE):
        print(f"  {entry['size']:>12d}  {entry['key']}")


[gsm8k] waiting for pruning_artifacts/20260506T144103Z-b08812/wanda_stats.pt ...
[humaneval] waiting for pruning_artifacts/20260506T144135Z-991a93/wanda_stats.pt ...
[arc_challenge] waiting for pruning_artifacts/20260506T144208Z-122c3c/wanda_stats.pt ...
[humaneval] DONE — wanda_stats.pt size=25,396,473 bytes
[arc_challenge] DONE — wanda_stats.pt size=25,396,473 bytes
[gsm8k] DONE — wanda_stats.pt size=25,396,473 bytes

All calibrations complete in 30.1 min.

[gsm8k] s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144103Z-b08812/
        184232  pruning_artifacts/20260506T144103Z-b08812/_logs/userdata.log
        114080  pruning_artifacts/20260506T144103Z-b08812/code/repo.tar.gz
           871  pruning_artifacts/20260506T144103Z-b08812/manifest.json
           390  pruning_artifacts/20260506T144103Z-b08812/run_metadata.json
        218781  pruning_artifacts/20260506T144103Z-b08812/split.json
      25396473  pruning_artifacts/20260506T144103Z-b08812/wanda_stats.pt



## Inspect manifests and splits

Downloads `manifest.json` and `split.json` for each artifact to confirm the
calibration completed correctly and the train/test counts look reasonable.


In [7]:
import boto3

session = boto3.session.Session(profile_name=AWS_PROFILE)
s3 = session.client("s3")

manifests = {}
for label, launched in calibration_launches:
    prefix = f"pruning_artifacts/{launched.run_id}"
    manifest = json.loads(
        s3.get_object(Bucket=RESULTS_BUCKET, Key=f"{prefix}/manifest.json")["Body"].read()
    )
    split = json.loads(
        s3.get_object(Bucket=RESULTS_BUCKET, Key=f"{prefix}/split.json")["Body"].read()
    )
    manifests[label] = manifest
    print(f"=== [{label}] manifest ===")
    print(json.dumps({
        k: v for k, v in manifest.items()
        if k not in ("package_versions", "host_metadata")
    }, indent=2))
    print(f"  train_ids[:3]: {split['train_task_ids'][:3]}")
    print(f"  test_ids[:3]:  {split['test_task_ids'][:3]}")
    print()


=== [gsm8k] manifest ===
{
  "schema_version": "1",
  "run_id": "20260506T144103Z-b08812",
  "base_model_id": "Qwen/Qwen2-72B",
  "dataset_spec": "math:gsm8k:main:train+test",
  "task_adapter": "math",
  "pruning_levels": [
    0.0,
    20.0,
    40.0,
    60.0,
    80.0
  ],
  "split_seed": 65320,
  "train_frac": 0.8,
  "max_calibration_samples": null,
  "max_calibration_tokens": 512,
  "num_train": 7473,
  "num_test": 1319,
  "explicit_train_ids": null,
  "explicit_test_ids": null,
  "host": "ip-172-31-40-167",
  "ended_at_utc": "2026-05-06T15:32:11.648095+00:00",
  "artifact_paths": {
    "wanda_stats": "wanda_stats.pt",
    "split": "split.json",
    "manifest": "manifest.json",
    "run_metadata": "run_metadata.json"
  }
}
  train_ids[:3]: ['gsm8k/train/00000', 'gsm8k/train/00001', 'gsm8k/train/00002']
  test_ids[:3]:  ['gsm8k/test/00000', 'gsm8k/test/00001', 'gsm8k/test/00002']

=== [humaneval] manifest ===
{
  "schema_version": "1",
  "run_id": "20260506T144135Z-991a93",
  "base

## Persist artifact URIs

Writes all three `PRUNING_ARTIFACT_URI` values to `experiment_config.json`
in this notebook directory. Notebook `02_eval_all.ipynb` reads this file at
startup so it can launch the evaluation runs without manual copy-paste.


In [11]:
import os
experiment_config_path = Path(os.getcwd()) / "experiment_config.json"

calibration_artifact_uris = {
    label: launched.results_uri
    for label, launched in calibration_launches
}

# Load existing config if present (to preserve keys written by later notebooks).
if experiment_config_path.exists():
    existing = json.loads(experiment_config_path.read_text(encoding="utf-8"))
else:
    existing = {}

existing["calibration_artifact_uris"] = calibration_artifact_uris
experiment_config_path.write_text(
    json.dumps(existing, indent=2), encoding="utf-8"
)

print("Wrote", experiment_config_path)
print(json.dumps(calibration_artifact_uris, indent=2))


Wrote /var/home/three-kingdoms/work/pruning-metrics/notebooks/experiment/experiment_config.json
{
  "gsm8k": "s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144103Z-b08812/",
  "humaneval": "s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144135Z-991a93/",
  "arc_challenge": "s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144208Z-122c3c/"
}
